In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Dataset
import numpy as np
import os
import glob
import copy
import random
import csv
import time
import tenseal as ts  # 真实同态加密库
from PIL import Image

# ==========================================
# 0. 全局配置
# ==========================================
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
SAVE_DIR = 'Ultimate_FL_CKKS_Results'
os.makedirs(SAVE_DIR, exist_ok=True)

def set_seed(seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    if DEVICE == 'cuda':
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.benchmark = True
set_seed(42)

IMG_SIZE = 128
PATCH_SIZE = 16
EMBED_DIM = 256
NUM_HEADS = 4
NUM_CLASSES_PRETRAIN = 10 
NUM_CLASSES_FINETUNE = 6  
FED_ROUNDS = 50

print(f"🚀 启动终极联邦学习任务 | 计算设备: {DEVICE}")

# ==========================================
# 1. 同态加密 (CKKS) 上下文初始化
# ==========================================
def setup_tenseal_context():
    print("🔒 正在初始化 TenSEAL CKKS 同态加密上下文...")
    context = ts.context(
        ts.SCHEME_TYPE.CKKS,
        poly_modulus_degree=8192,
        coeff_mod_bit_sizes=[60, 40, 40, 60]
    )
    context.global_scale = 2**40
    context.generate_galois_keys()
    print("✅ 同态加密环境就绪！")
    return context

HE_CONTEXT = setup_tenseal_context()

# ==========================================
# 2. 模型架构定义
# ==========================================
class ForwardDefense(nn.Module):
    def __init__(self, std=0.0):
        super().__init__()
        self.noise_std = std
    def forward(self, x):
        if self.training and self.noise_std > 0:
            return x + torch.randn_like(x) * self.noise_std
        return x

class StandardBlock(nn.Module):
    def __init__(self, dim, num_heads):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(dim, num_heads, batch_first=True)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(nn.Linear(dim, dim * 4), nn.GELU(), nn.Linear(dim * 4, dim))
    def forward(self, x):
        x = x + self.attn(self.norm1(x), self.norm1(x), self.norm1(x))[0]
        x = x + self.mlp(self.norm2(x))
        return x

class ClientModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.proj = nn.Conv2d(3, EMBED_DIM, kernel_size=PATCH_SIZE, stride=PATCH_SIZE)
        num_patches = (IMG_SIZE // PATCH_SIZE) ** 2
        self.pos_embed = nn.Parameter(torch.randn(1, num_patches, EMBED_DIM) * .02)
        self.blocks = nn.ModuleList([StandardBlock(EMBED_DIM, NUM_HEADS) for _ in range(2)])
        self.defense = ForwardDefense(std=0.0)
    def forward(self, x):
        x = self.proj(x).flatten(2).transpose(1, 2)
        x = x + self.pos_embed
        for blk in self.blocks: x = blk(x)
        return self.defense(x)

class ServerModel(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.blocks = nn.ModuleList([StandardBlock(EMBED_DIM, NUM_HEADS) for _ in range(2)])
        self.norm = nn.LayerNorm(EMBED_DIM)
        self.head = nn.Linear(EMBED_DIM, num_classes)
    def forward(self, x):
        for blk in self.blocks: x = blk(x)
        x = self.norm(x)
        return self.head(x.mean(dim=1))

# ==========================================
# 3. 聚合机制与梯度防御
# ==========================================
def apply_gradient_defense(grad, prune_ratio=0.5, noise_std=1e-3):
    """梯度剪枝与差分隐私加噪"""
    if grad is None: return None
    g = grad.detach().clone()
    k = int(g.numel() * (1 - prune_ratio))
    if k > 0:
        thresh = torch.kthvalue(g.abs().flatten(), g.numel() - k + 1).values
        mask = g.abs() >= thresh
        g = g * mask
    return g + torch.randn_like(g) * noise_std

def fed_avg_plaintext(global_model, client_models, weights):
    """无防御状态下的明文聚合 (极速)"""
    with torch.no_grad():
        w_avg = global_model.state_dict()
        for k in w_avg.keys():
            weighted_params = [c.state_dict()[k].float() * w for c, w in zip(client_models, weights)]
            w_avg[k] = torch.stack(weighted_params, dim=0).sum(dim=0)
        global_model.load_state_dict(w_avg)

def real_secure_fed_avg(global_model, client_models, weights):
    """基于 TenSEAL 的真实同态加密聚合 (极慢，但密码学安全)"""
    with torch.no_grad():
        global_dict = global_model.state_dict()
        for k in global_dict.keys():
            param_shape = global_dict[k].shape
            
            # 1. Client端加密
            encrypted_clients = []
            for c in client_models:
                flat_data = c.state_dict()[k].flatten().tolist()
                enc_vector = ts.ckks_vector(HE_CONTEXT, flat_data)
                encrypted_clients.append(enc_vector)
                
            # 2. Server端密文加权聚合
            enc_avg = encrypted_clients[0] * weights[0]
            for i in range(1, len(client_models)):
                enc_avg += encrypted_clients[i] * weights[i]
                
            # 3. Client端解密回填
            decrypted_data = enc_avg.decrypt()
            new_param = torch.tensor(decrypted_data).reshape(param_shape).to(DEVICE)
            global_dict[k] = new_param
            
        global_model.load_state_dict(global_dict)

# ==========================================
# 4. 数据分布与加载
# ==========================================
class NEUDataset(Dataset):
    def __init__(self, files, labels, transform=None):
        self.files = files; self.labels = labels; self.transform = transform
    def __len__(self): return len(self.files)
    def __getitem__(self, idx):
        img = Image.open(self.files[idx]).convert('RGB')
        if self.transform: img = self.transform(img)
        return img, self.labels[idx]

def generate_fl_data(root, mode='iid', alpha=1.0, num_clients=3):
    CLASSES = ['crazing', 'inclusion', 'patches', 'pitted', 'rolled', 'scratches']
    cls_map = {c: i for i, c in enumerate(CLASSES)}
    files, labels = [], []
    for ext in ['*.jpg', '*.png', '*.bmp']:
        for path in glob.glob(os.path.join(root, ext)):
            name = os.path.basename(path).lower()
            for c in CLASSES:
                if c in name or (c=='scratches' and 'sc_' in name):
                    files.append(path); labels.append(cls_map[c]); break
    files, labels = np.array(files), np.array(labels)
    indices = np.random.permutation(len(files))
    split_idx = int(len(files) * 0.8)
    train_f, train_l = files[indices[:split_idx]], labels[indices[:split_idx]]
    test_f, test_l = files[indices[split_idx:]], labels[indices[split_idx:]]
    
    cf, cl = [[] for _ in range(num_clients)], [[] for _ in range(num_clients)]
    if mode == 'iid':
        splits = np.array_split(np.random.permutation(len(train_f)), num_clients)
        for i in range(num_clients): cf[i], cl[i] = train_f[splits[i]].tolist(), train_l[splits[i]].tolist()
    elif mode == 'weak_non_iid':
        for c in range(NUM_CLASSES_FINETUNE):
            idx_c = np.where(train_l == c)[0]; np.random.shuffle(idx_c)
            prop = np.cumsum(np.random.dirichlet(np.repeat(alpha, num_clients))) * len(idx_c)
            splits = np.split(idx_c, prop.astype(int)[:-1])
            for i in range(num_clients): cf[i].extend(train_f[splits[i]].tolist()); cl[i].extend(train_l[splits[i]].tolist())
    elif mode == 'pathological':
        targets = [[0, 1], [2, 3], [4, 5]]
        for i in range(num_clients):
            idx = [idx for idx, label in enumerate(train_l) if label in targets[i]]
            cf[i], cl[i] = train_f[idx].tolist(), train_l[idx].tolist()

    tf = transforms.Compose([transforms.Resize((IMG_SIZE, IMG_SIZE)), transforms.RandomHorizontalFlip(), transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])
    test_tf = transforms.Compose([transforms.Resize((IMG_SIZE, IMG_SIZE)), transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])
    loaders = [DataLoader(NEUDataset(cf[i], cl[i], tf), batch_size=32, shuffle=True) for i in range(num_clients)]
    test_loader = DataLoader(NEUDataset(test_f.tolist(), test_l.tolist(), test_tf), batch_size=64, shuffle=False)
    return loaders, test_loader

# ==========================================
# 5. 带时间戳的预训练沿途下蛋
# ==========================================
def train_pretrain_checkpoints(target_epochs=[3, 5, 10, 20]):
    max_epochs = max(target_epochs)
    print("\n" + "★"*60)
    print(f"[阶段 1] CIFAR-10 集中式预训练 (共 {max_epochs} 轮)")
    print(f"目标保存节点: {target_epochs}")
    print("★"*60)
    
    tf = transforms.Compose([transforms.Resize((IMG_SIZE, IMG_SIZE)), transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])
    trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=tf)
    loader = DataLoader(trainset, batch_size=64, shuffle=True)
    
    c_m = ClientModel().to(DEVICE); s_m = ServerModel(NUM_CLASSES_PRETRAIN).to(DEVICE)
    opt = optim.AdamW(list(c_m.parameters()) + list(s_m.parameters()), lr=1e-3)
    crit = nn.CrossEntropyLoss()
    
    checkpoints = {}
    total_start_time = time.time()
    
    for ep in range(1, max_epochs + 1):
        epoch_start = time.time()
        c_m.train(); s_m.train(); loss_sum = 0
        for imgs, lbls in loader:
            imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
            opt.zero_grad(); loss = crit(s_m(c_m(imgs)), lbls)
            loss.backward(); opt.step(); loss_sum += loss.item()
            
        epoch_time = time.time() - epoch_start
        print(f"  > Pretrain Epoch {ep:02d} | Avg Loss: {loss_sum/len(loader):.4f} | 耗时: {epoch_time:.2f} 秒")
        
        if ep in target_epochs:
            checkpoints[ep] = (copy.deepcopy(c_m), copy.deepcopy(s_m))
            print(f"    📦 已保存权重: 第 {ep} 轮 (累计耗时: {(time.time()-total_start_time)/60:.2f} 分钟)")
            
    print(f"✅ 预训练结束！总耗时: {(time.time()-total_start_time)/60:.2f} 分钟\n")
    return checkpoints

# ==========================================
# 6. 联邦微调核心流程 (支持同态加密切换)
# ==========================================
def run_fl_experiment(mode, loaders, test_loader, use_defense, pretrain_weights, pre_ep, exp_idx, total_exp):
    def_str = "with_defense" if use_defense else "no_defense"
    print(f"\n" + "="*70)
    print(f"💠 [实验 {exp_idx}/{total_exp}] 分布: {mode.upper()} | 预训练: {pre_ep}轮 | 防御: {def_str.upper()}")
    if use_defense: print("⚠️ 注意：已开启 TenSEAL 同态加密，单轮耗时将显著增加！")
    print("="*70)
    
    c_weights = [len(l.dataset)/sum([len(x.dataset) for x in loaders]) for l in loaders]
    
    g_client = copy.deepcopy(pretrain_weights[0])
    g_server = ServerModel(NUM_CLASSES_FINETUNE).to(DEVICE)
    g_server.blocks.load_state_dict(pretrain_weights[1].blocks.state_dict())
    
    clients = [copy.deepcopy(g_client) for _ in range(3)]
    servers = [copy.deepcopy(g_server) for _ in range(3)]
    
    # 开启/关闭前向加噪
    for c in clients: c.defense.noise_std = 0.001 if use_defense else 0.0
    
    opt_c = [optim.AdamW(c.parameters(), lr=1e-4) for c in clients]
    opt_s = [optim.AdamW(s.parameters(), lr=1e-4) for s in servers]
    crit = nn.CrossEntropyLoss()
    
    history = {'loss': [], 'acc': []}; best_acc = 0.0

    for r in range(1, FED_ROUNDS + 1):
        round_start = time.time()
        r_losses = []
        
        # --- 边缘端本地训练 ---
        for i in range(3):
            clients[i].train(); servers[i].train(); ep_loss = 0
            for imgs, lbls in loaders[i]:
                imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
                opt_c[i].zero_grad(); opt_s[i].zero_grad()
                z = clients[i](imgs); zp = z.detach().clone().requires_grad_(True)
                out = servers[i](zp); loss = crit(out, lbls); loss.backward()
                
                # 开启/关闭后向梯度防御
                safe_g = apply_gradient_defense(zp.grad) if use_defense else zp.grad
                z.backward(safe_g); opt_c[i].step(); opt_s[i].step(); ep_loss += loss.item()
            r_losses.append(ep_loss/len(loaders[i]))
        
        avg_l = sum(r_losses)/3; history['loss'].append(avg_l)
        
        # --- 云端联邦聚合 (明文 vs 同态加密) ---
        if use_defense:
            real_secure_fed_avg(g_client, clients, c_weights)
            real_secure_fed_avg(g_server, servers, c_weights)
        else:
            fed_avg_plaintext(g_client, clients, c_weights)
            fed_avg_plaintext(g_server, servers, c_weights)
            
        for i in range(3): 
            clients[i].load_state_dict(g_client.state_dict())
            servers[i].load_state_dict(g_server.state_dict())
        
        # --- 评估 ---
        g_client.eval(); g_server.eval(); corr, total = 0, 0
        with torch.no_grad():
            for imgs, lbls in test_loader:
                imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
                corr += (g_server(g_client(imgs)).argmax(1) == lbls).sum().item(); total += lbls.size(0)
        acc = 100 * corr / total; history['acc'].append(acc)
        
        round_time = time.time() - round_start
        status = f"  Round {r:02d}/{FED_ROUNDS} | Loss: {avg_l:.4f} | Acc: {acc:.2f}% | 耗时: {round_time:.1f}s"
        
        if acc > best_acc:
            best_acc = acc
            torch.save(g_client.state_dict(), f'{SAVE_DIR}/best_client_{mode}_pre{pre_ep}_{def_str}.pth')
            torch.save(g_server.state_dict(), f'{SAVE_DIR}/best_server_{mode}_pre{pre_ep}_{def_str}.pth')
            status += " ⭐ [Best Saved]"
        print(status)

    csv_fn = f'history_{mode}_pre{pre_ep}_{def_str}.csv'
    with open(f'{SAVE_DIR}/{csv_fn}', 'w', newline='') as f:
        writer = csv.writer(f); writer.writerow(['Round', 'Loss', 'Accuracy'])
        for idx, (l, a) in enumerate(zip(history['loss'], history['acc']), 1): writer.writerow([idx, l, a])
    print(f"  🏁 实验 {exp_idx} 完成! 最佳精度: {best_acc:.2f}% | 记录存至: {csv_fn}")

# ==========================================
# 7. 主程序流控制
# ==========================================
if __name__ == '__main__':
    # ⚠️ 请修改为你的服务器真实图片路径
    DATA_ROOT = r'/root/autodl-tmp/exp2/data/NEU/IMAGES' 
    
    # 实验维度配置 (如觉得 HE 太慢，可减少 pre_epochs_list)
    pre_epochs_list = [3, 5, 10, 20]
    distributions = ['iid', 'weak_non_iid', 'pathological'] # 包含了所有三种分布
    defenses = [False, True]
    
    total_experiments = len(distributions) * len(pre_epochs_list) * len(defenses)
    
    # 1. 启动带计时的集中式预训练
    pretrain_checkpoints = train_pretrain_checkpoints(target_epochs=pre_epochs_list)
    
    # 2. 正式开展矩阵消融实验
    exp_count = 1
    for dist in distributions:
        print(f"\n📦 正在切割并锁定 [{dist.upper()}] 分布的训练/测试集...")
        current_loaders, current_test_loader = generate_fl_data(DATA_ROOT, mode=dist)
        
        for pre_ep in pre_epochs_list:
            for defense in defenses:
                weights_for_this_exp = pretrain_checkpoints[pre_ep]
                run_fl_experiment(
                    mode=dist, 
                    loaders=current_loaders, 
                    test_loader=current_test_loader, 
                    use_defense=defense, 
                    pretrain_weights=weights_for_this_exp, 
                    pre_ep=pre_ep, 
                    exp_idx=exp_count, 
                    total_exp=total_experiments
                )
                exp_count += 1
                
    print("\n" + "✅"*25 + "\n所有消融实验已全部圆满结束！\n" + "✅"*25)
    print(f"所有 CSV 与 PTH 文件存放于: ./{SAVE_DIR}/")

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Dataset
import numpy as np
import os
import glob
import copy
import random
import csv
import time
import tenseal as ts
from PIL import Image

# ==========================================
# 0. 全局配置
# ==========================================
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
SAVE_DIR = 'Single_Test_Results'
os.makedirs(SAVE_DIR, exist_ok=True)

def set_seed(seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    if DEVICE == 'cuda':
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.benchmark = True
set_seed(42)

IMG_SIZE = 128
PATCH_SIZE = 16
EMBED_DIM = 256
NUM_HEADS = 4
NUM_CLASSES_PRETRAIN = 10 
NUM_CLASSES_FINETUNE = 6  
FED_ROUNDS = 50

print(f"🚀 启动单点测试任务 | 设备: {DEVICE}")

# ==========================================
# 1. 同态加密 (CKKS) 上下文初始化
# ==========================================
print("🔒 正在初始化 TenSEAL CKKS 环境 (这可能会导致内存占用增加)...")
HE_CONTEXT = ts.context(
    ts.SCHEME_TYPE.CKKS,
    poly_modulus_degree=8192,
    coeff_mod_bit_sizes=[60, 40, 40, 60]
)
HE_CONTEXT.global_scale = 2**40
HE_CONTEXT.generate_galois_keys()
print("✅ 同态加密环境就绪！")

# ==========================================
# 2. 模型定义
# ==========================================
class ForwardDefense(nn.Module):
    def __init__(self, std=0.0):
        super().__init__()
        self.noise_std = std
    def forward(self, x):
        if self.training and self.noise_std > 0:
            return x + torch.randn_like(x) * self.noise_std
        return x

class StandardBlock(nn.Module):
    def __init__(self, dim, num_heads):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(dim, num_heads, batch_first=True)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(nn.Linear(dim, dim * 4), nn.GELU(), nn.Linear(dim * 4, dim))
    def forward(self, x):
        x = x + self.attn(self.norm1(x), self.norm1(x), self.norm1(x))[0]
        x = x + self.mlp(self.norm2(x))
        return x

class ClientModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.proj = nn.Conv2d(3, EMBED_DIM, kernel_size=PATCH_SIZE, stride=PATCH_SIZE)
        num_patches = (IMG_SIZE // PATCH_SIZE) ** 2
        self.pos_embed = nn.Parameter(torch.randn(1, num_patches, EMBED_DIM) * .02)
        self.blocks = nn.ModuleList([StandardBlock(EMBED_DIM, NUM_HEADS) for _ in range(2)])
        self.defense = ForwardDefense(std=0.0)
    def forward(self, x):
        x = self.proj(x).flatten(2).transpose(1, 2)
        x = x + self.pos_embed
        for blk in self.blocks: x = blk(x)
        return self.defense(x)

class ServerModel(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.blocks = nn.ModuleList([StandardBlock(EMBED_DIM, NUM_HEADS) for _ in range(2)])
        self.norm = nn.LayerNorm(EMBED_DIM)
        self.head = nn.Linear(EMBED_DIM, num_classes)
    def forward(self, x):
        for blk in self.blocks: x = blk(x)
        x = self.norm(x)
        return self.head(x.mean(dim=1))

# ==========================================
# 3. 数据与防御机制
# ==========================================
def apply_gradient_defense(grad, prune_ratio=0.5, noise_std=1e-3):
    if grad is None: return None
    g = grad.detach().clone()
    k = int(g.numel() * (1 - prune_ratio))
    if k > 0:
        thresh = torch.kthvalue(g.abs().flatten(), g.numel() - k + 1).values
        mask = g.abs() >= thresh
        g = g * mask
    return g + torch.randn_like(g) * noise_std

def real_secure_fed_avg(global_model, client_models, weights):
    with torch.no_grad():
        global_dict = global_model.state_dict()
        for k in global_dict.keys():
            param_shape = global_dict[k].shape
            encrypted_clients = []
            for c in client_models:
                flat_data = c.state_dict()[k].flatten().tolist()
                enc_vector = ts.ckks_vector(HE_CONTEXT, flat_data)
                encrypted_clients.append(enc_vector)
                
            enc_avg = encrypted_clients[0] * weights[0]
            for i in range(1, len(client_models)):
                enc_avg += encrypted_clients[i] * weights[i]
                
            decrypted_data = enc_avg.decrypt()
            new_param = torch.tensor(decrypted_data).reshape(param_shape).to(DEVICE)
            global_dict[k] = new_param
            
        global_model.load_state_dict(global_dict)

class NEUDataset(Dataset):
    def __init__(self, files, labels, transform=None):
        self.files = files; self.labels = labels; self.transform = transform
    def __len__(self): return len(self.files)
    def __getitem__(self, idx):
        img = Image.open(self.files[idx]).convert('RGB')
        if self.transform: img = self.transform(img)
        return img, self.labels[idx]

def generate_pathological_data(root, num_clients=3):
    print("\n📦 正在切割并锁定 [PATHOLOGICAL] 极端非独立同分布数据...")
    CLASSES = ['crazing', 'inclusion', 'patches', 'pitted', 'rolled', 'scratches']
    cls_map = {c: i for i, c in enumerate(CLASSES)}
    files, labels = [], []
    for ext in ['*.jpg', '*.png', '*.bmp']:
        for path in glob.glob(os.path.join(root, ext)):
            name = os.path.basename(path).lower()
            for c in CLASSES:
                if c in name or (c=='scratches' and 'sc_' in name):
                    files.append(path); labels.append(cls_map[c]); break
    files, labels = np.array(files), np.array(labels)
    indices = np.random.permutation(len(files))
    split_idx = int(len(files) * 0.8)
    train_f, train_l = files[indices[:split_idx]], labels[indices[:split_idx]]
    test_f, test_l = files[indices[split_idx:]], labels[indices[split_idx:]]
    
    cf, cl = [[] for _ in range(num_clients)], [[] for _ in range(num_clients)]
    targets = [[0, 1], [2, 3], [4, 5]]
    for i in range(num_clients):
        idx = [idx for idx, label in enumerate(train_l) if label in targets[i]]
        cf[i], cl[i] = train_f[idx].tolist(), train_l[idx].tolist()

    tf = transforms.Compose([transforms.Resize((IMG_SIZE, IMG_SIZE)), transforms.RandomHorizontalFlip(), transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])
    test_tf = transforms.Compose([transforms.Resize((IMG_SIZE, IMG_SIZE)), transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])
    loaders = [DataLoader(NEUDataset(cf[i], cl[i], tf), batch_size=32, shuffle=True) for i in range(num_clients)]
    test_loader = DataLoader(NEUDataset(test_f.tolist(), test_l.tolist(), test_tf), batch_size=64, shuffle=False)
    return loaders, test_loader

# ==========================================
# 4. 专属的 10轮 预训练
# ==========================================
def train_10ep_pretrain():
    print("\n" + "★"*50)
    print("[阶段 1] CIFAR-10 集中式预训练 (执行 10 轮)")
    print("★"*50)
    tf = transforms.Compose([transforms.Resize((IMG_SIZE, IMG_SIZE)), transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])
    trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=tf)
    loader = DataLoader(trainset, batch_size=64, shuffle=True)
    
    c_m = ClientModel().to(DEVICE); s_m = ServerModel(NUM_CLASSES_PRETRAIN).to(DEVICE)
    opt = optim.AdamW(list(c_m.parameters()) + list(s_m.parameters()), lr=1e-3)
    crit = nn.CrossEntropyLoss()
    
    total_start = time.time()
    for ep in range(1, 11):
        ep_start = time.time()
        c_m.train(); s_m.train(); loss_sum = 0
        for imgs, lbls in loader:
            imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
            opt.zero_grad(); loss = crit(s_m(c_m(imgs)), lbls)
            loss.backward(); opt.step(); loss_sum += loss.item()
        print(f"  > Pretrain Epoch {ep:02d} | Avg Loss: {loss_sum/len(loader):.4f} | 耗时: {time.time()-ep_start:.2f}s")
        
    print(f"✅ 10轮预训练完成！总耗时: {(time.time()-total_start)/60:.2f} 分钟\n")
    return c_m, s_m

# ==========================================
# 5. 单组目标实验运行
# ==========================================
if __name__ == '__main__':
    # ⚠️ 请确保修改为真实的图片路径
    DATA_ROOT = r'/root/autodl-tmp/exp2/data/NEU/IMAGES' 
    
    # 1. 跑 10 轮预训练
    pre_c, pre_s = train_10ep_pretrain()
    
    # 2. 生成 Pathological 数据
    loaders, test_loader = generate_pathological_data(DATA_ROOT)
    c_weights = [len(l.dataset)/sum([len(x.dataset) for x in loaders]) for l in loaders]
    for i in range(3): print(f"  - Client {i+1} 样本数: {len(loaders[i].dataset)}")
    
    # 3. 初始化联邦环境
    print("\n" + "="*60)
    print("💠 [联邦测试] 分布: PATHOLOGICAL | 预训练: 10轮 | 防御: 开启(CKKS)")
    print("⚠️ 警告：真实同态加密聚合极度消耗CPU计算资源，请留意单轮耗时！")
    print("="*60)
    
    g_client = copy.deepcopy(pre_c)
    g_server = ServerModel(NUM_CLASSES_FINETUNE).to(DEVICE)
    g_server.blocks.load_state_dict(pre_s.blocks.state_dict())
    
    clients = [copy.deepcopy(g_client) for _ in range(3)]
    servers = [copy.deepcopy(g_server) for _ in range(3)]
    
    # 开启前向特征加噪
    for c in clients: c.defense.noise_std = 0.001 
    
    opt_c = [optim.AdamW(c.parameters(), lr=1e-4) for c in clients]
    opt_s = [optim.AdamW(s.parameters(), lr=1e-4) for s in servers]
    crit = nn.CrossEntropyLoss()
    
    history = {'loss': [], 'acc': []}; best_acc = 0.0
    total_fl_start = time.time()

    # 4. 执行 50 轮联邦学习
    for r in range(1, FED_ROUNDS + 1):
        round_start = time.time()
        r_losses = []
        
        # 本地训练 + 后向防御
        for i in range(3):
            clients[i].train(); servers[i].train(); ep_loss = 0
            for imgs, lbls in loaders[i]:
                imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
                opt_c[i].zero_grad(); opt_s[i].zero_grad()
                z = clients[i](imgs); zp = z.detach().clone().requires_grad_(True)
                out = servers[i](zp); loss = crit(out, lbls); loss.backward()
                
                safe_g = apply_gradient_defense(zp.grad) # 开启后向梯度防御
                z.backward(safe_g); opt_c[i].step(); opt_s[i].step(); ep_loss += loss.item()
            r_losses.append(ep_loss/len(loaders[i]))
        
        avg_l = sum(r_losses)/3; history['loss'].append(avg_l)
        
        # 🌟 真实 CKKS 同态安全聚合
        real_secure_fed_avg(g_client, clients, c_weights)
        real_secure_fed_avg(g_server, servers, c_weights)
            
        for i in range(3): 
            clients[i].load_state_dict(g_client.state_dict())
            servers[i].load_state_dict(g_server.state_dict())
        
        # 评估
        g_client.eval(); g_server.eval(); corr, total = 0, 0
        with torch.no_grad():
            for imgs, lbls in test_loader:
                imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
                corr += (g_server(g_client(imgs)).argmax(1) == lbls).sum().item(); total += lbls.size(0)
        acc = 100 * corr / total; history['acc'].append(acc)
        
        round_time = time.time() - round_start
        status = f"  Round {r:02d}/{FED_ROUNDS} | Loss: {avg_l:.4f} | Acc: {acc:.2f}% | 轮耗时: {round_time:.1f} 秒"
        
        if acc > best_acc:
            best_acc = acc
            torch.save(g_client.state_dict(), f'{SAVE_DIR}/best_client_patho_pre10_with_defense.pth')
            torch.save(g_server.state_dict(), f'{SAVE_DIR}/best_server_patho_pre10_with_defense.pth')
            status += " ⭐ [Best Saved]"
        print(status)

    csv_fn = 'history_pathological_pre10_with_defense.csv'
    with open(f'{SAVE_DIR}/{csv_fn}', 'w', newline='') as f:
        writer = csv.writer(f); writer.writerow(['Round', 'Loss', 'Accuracy'])
        for idx, (l, a) in enumerate(zip(history['loss'], history['acc']), 1): writer.writerow([idx, l, a])
        
    print("\n" + "="*60)
    print(f"🏁 单点测试完成! 最佳精度: {best_acc:.2f}%")
    print(f"⏱️ 联邦微调总耗时: {(time.time()-total_fl_start)/60:.2f} 分钟")
    print(f"记录已存至: ./{SAVE_DIR}/{csv_fn}")
    print("="*60)

🚀 启动单点测试任务 | 设备: cuda
🔒 正在初始化 TenSEAL CKKS 环境 (这可能会导致内存占用增加)...
✅ 同态加密环境就绪！

★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★
[阶段 1] CIFAR-10 集中式预训练 (执行 10 轮)
★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★
Files already downloaded and verified
  > Pretrain Epoch 01 | Avg Loss: 1.8572 | 耗时: 32.28s
  > Pretrain Epoch 02 | Avg Loss: 1.6395 | 耗时: 29.90s
  > Pretrain Epoch 03 | Avg Loss: 1.5694 | 耗时: 32.31s
  > Pretrain Epoch 04 | Avg Loss: 1.5281 | 耗时: 31.71s
  > Pretrain Epoch 05 | Avg Loss: 1.4831 | 耗时: 31.93s
  > Pretrain Epoch 06 | Avg Loss: 1.4634 | 耗时: 31.12s
  > Pretrain Epoch 07 | Avg Loss: 1.4195 | 耗时: 32.23s
  > Pretrain Epoch 08 | Avg Loss: 1.3853 | 耗时: 31.60s
  > Pretrain Epoch 09 | Avg Loss: 1.3790 | 耗时: 28.78s
  > Pretrain Epoch 10 | Avg Loss: 1.3534 | 耗时: 29.12s
✅ 10轮预训练完成！总耗时: 5.18 分钟


📦 正在切割并锁定 [PATHOLOGICAL] 极端非独立同分布数据...
  - Client 1 样本数: 494
  - Client 2 样本数: 480
  - Client 3 样本数: 466

💠 [联邦测试] 分布: PATHOLOGICAL | 预训练: 10轮 | 防御: 开启(CKKS)
⚠️ 警告：真实同态加密聚合极度消耗CPU计算资源，